In [0]:
# Paths to Delta tables
users_path = "/Volumes/project_2/datalake/silver/user_events/"
transactions_path = "/Volumes/project_2/datalake/silver/transactions/"
customers_path = "/Volumes/project_2/datalake/landing_zone/customers"
products_path = "/Volumes/project_2/datalake/landing_zone/products"
gold_dir = "/Volumes/project_2/datalake/gold/"

# Read Delta data
users_df = spark.read.format("delta").load(users_path)
transactions_df = spark.read.format("delta").load(transactions_path)

# Register as temp views
users_df.createOrReplaceTempView("users")
transactions_df.createOrReplaceTempView("transactions")

# Create and write gold layer tables
# dim_customer
customers_df = spark.read.option("multiLine", "true").json(customers_path)
customers_df.createOrReplaceTempView("customers_json")
dim_customer = spark.sql("""
    SELECT user_id, email, first_name, last_name, CAST(registration_date AS DATE) AS registration_date, account_type, CAST(date_of_birth AS DATE) AS date_of_birth, loyalty_points, state
    FROM customers_json
""")
dim_customer.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_customer")

# dim_product
products_df = spark.read.option("multiLine", "true").json(products_path)
products_df.createOrReplaceTempView("products_json")
dim_product = spark.sql("""
    SELECT DISTINCT product_id, product_name, description, category, subcategory, brand, manufacturer, CAST(created_date AS DATE) AS created_date, is_active
    FROM products_json
""")
dim_product.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_product")

# dim_date
dim_date = spark.sql("""
    SELECT DISTINCT event_date
    FROM (
        SELECT CAST(timestamp AS DATE) AS event_date FROM transactions
        UNION
        SELECT CAST(timestamp AS DATE) AS event_date FROM users
    )
""")
dim_date.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_date")

# fact_transactions with payment_method and flattened billing/shipping address
fact_transactions = spark.sql("""
    SELECT
        transaction_id,
        user_id,
        transaction_type,
        CAST(timestamp AS DATE) AS event_date,
        status,
        currency,
        payment_method,
        product_id,
        quantity,
        unit_price,
        total,
        billing_street,
        billing_city,
        billing_state,
        billing_zip_code,
        billing_country,
        shipping_street,
        shipping_city,
        shipping_state,
        shipping_zip_code,
        shipping_country
    FROM transactions
""")
fact_transactions.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}fact_transactions")

# fact_user_activity
fact_user_activity = spark.sql("""
    SELECT user_id, session_id, event_id, event_type, CAST(timestamp AS DATE) AS event_date, page, device, browser, country, city, search_query, element_id, product_id, quantity
    FROM users
""")
fact_user_activity.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}fact_user_activity")

display(dim_customer)
display(dim_product)
display(dim_date)
display(fact_transactions)
display(fact_user_activity)

In [0]:
# Register gold tables as temp views
spark.read.format("delta").load(f"{gold_dir}dim_customer").createOrReplaceTempView("dim_customer")
spark.read.format("delta").load(f"{gold_dir}fact_transactions").createOrReplaceTempView("fact_transactions")
spark.read.format("delta").load(f"{gold_dir}fact_user_activity").createOrReplaceTempView("fact_user_activity")
spark.read.format("delta").load(f"{gold_dir}dim_product").createOrReplaceTempView("dim_product")

v_order_features = spark.sql("""
WITH order_base AS (
    -- Aggregate to order level: one row per transaction_id
    SELECT
        transaction_id AS order_id,
        user_id,
        MIN(event_date) AS event_date,
        MIN(currency) AS currency,
        MIN(payment_method) AS payment_method,
        total AS order_total,
        billing_street,
        billing_city,
        billing_state,
        billing_zip_code,
        billing_country,
        shipping_street,
        shipping_city,
        shipping_state,
        shipping_zip_code,
        shipping_country,
        SUM(quantity) AS num_items,
        COUNT(DISTINCT product_id) AS num_distinct_products
    FROM fact_transactions
    GROUP BY transaction_id, user_id, total, billing_street, billing_city, billing_state, billing_zip_code, billing_country, shipping_street, shipping_city, shipping_state, shipping_zip_code, shipping_country
),
account_age AS (
    SELECT
        user_id,
        DATEDIFF(CURRENT_DATE(), registration_date) AS account_age_days
    FROM dim_customer
),
order_history_base AS (
    -- One row per order with metrics for computing historical stats
    SELECT
        transaction_id,
        user_id,
        event_date,
        status,
        transaction_type,
        SUM(unit_price * quantity) AS order_value
    FROM fact_transactions
    GROUP BY transaction_id, user_id, event_date, status, transaction_type
),
historical_with_windows AS (
    -- Compute historical metrics using window functions (excluding current row)
    SELECT
        transaction_id,
        user_id,
        event_date,
        COALESCE(SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) 
            OVER (PARTITION BY user_id ORDER BY event_date, transaction_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS previous_completed_purchases,
        COALESCE(SUM(CASE WHEN transaction_type = 'refund' THEN 1 ELSE 0 END)
            OVER (PARTITION BY user_id ORDER BY event_date, transaction_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS previous_returns,
        COALESCE(SUM(CASE WHEN transaction_type = 'chargeback' THEN 1 ELSE 0 END)
            OVER (PARTITION BY user_id ORDER BY event_date, transaction_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS previous_chargebacks,
        AVG(order_value)
            OVER (PARTITION BY user_id ORDER BY event_date, transaction_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS historical_avg_order_value,
        DATEDIFF(event_date, LAG(event_date) OVER (PARTITION BY user_id ORDER BY event_date, transaction_id)) AS days_since_prev_purchase,
        LAG(event_date) OVER (PARTITION BY user_id ORDER BY event_date, transaction_id) AS last_purchase_date
    FROM order_history_base
),
historical AS (
    SELECT
        transaction_id,
        user_id,
        previous_completed_purchases,
        previous_returns,
        previous_chargebacks,
        ROUND(historical_avg_order_value, 2) AS historical_avg_order_value,
        CASE WHEN last_purchase_date IS NOT NULL 
             THEN DATEDIFF(CURRENT_DATE(), last_purchase_date)
             ELSE NULL END AS days_since_last_purchase,
        CASE WHEN previous_completed_purchases > 0 
             THEN CAST(previous_returns AS DOUBLE) / previous_completed_purchases 
             ELSE 0 END AS user_return_rate
    FROM historical_with_windows
),
avg_purchase_interval AS (
    SELECT
        user_id,
        ROUND(AVG(days_since_prev_purchase), 2) AS avg_days_between_purchases
    FROM historical_with_windows
    WHERE days_since_prev_purchase IS NOT NULL
    GROUP BY user_id
),
user_sessions_daily AS (
    -- Get distinct sessions per user per date for session counting
    SELECT DISTINCT
        user_id,
        event_date,
        session_id
    FROM fact_user_activity
),
user_activity_aggregated AS (
    -- Aggregate daily user activity metrics
    SELECT
        user_id,
        event_date,
        COUNT(page) AS page_views,
        SUM(CASE WHEN event_type = 'cart_add' THEN 1 ELSE 0 END) AS cart_adds,
        SUM(CASE WHEN event_type = 'cart_remove' THEN 1 ELSE 0 END) AS cart_removals,
        SUM(CASE WHEN event_type = 'search' THEN 1 ELSE 0 END) AS searches
    FROM fact_user_activity
    GROUP BY user_id, event_date
),
user_activity_7d AS (
    -- Rolling 7-day window of user activity
    SELECT DISTINCT
        a.user_id,
        a.event_date,
        COUNT(s.session_id) 
            OVER (PARTITION BY a.user_id ORDER BY a.event_date 
                  RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS user_session_count_7d,
        SUM(a.page_views) 
            OVER (PARTITION BY a.user_id ORDER BY a.event_date 
                  RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS user_page_views_7d,
        SUM(a.cart_adds)
            OVER (PARTITION BY a.user_id ORDER BY a.event_date 
                  RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS user_cart_adds_7d,
        SUM(a.cart_removals)
            OVER (PARTITION BY a.user_id ORDER BY a.event_date 
                  RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS user_cart_removals_7d,
        SUM(a.searches)
            OVER (PARTITION BY a.user_id ORDER BY a.event_date 
                  RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS user_searches_7d
    FROM user_activity_aggregated a
    LEFT JOIN user_sessions_daily s ON a.user_id = s.user_id AND a.event_date = s.event_date
),
order_primary_category AS (
    -- Get primary category per order (category with highest value in that order)
    SELECT 
        t.transaction_id AS order_id,
        FIRST(p.category) AS primary_category
    FROM fact_transactions t
    LEFT JOIN dim_product p ON t.product_id = p.product_id
    GROUP BY t.transaction_id
    HAVING MAX(t.unit_price * t.quantity) IS NOT NULL
),
order_primary_device AS (
    -- Get primary device per order (most common device on that user+date)
    SELECT
        ob.order_id,
        FIRST(ua.device) AS primary_device
    FROM order_base ob
    LEFT JOIN fact_user_activity ua ON ob.user_id = ua.user_id AND ob.event_date = ua.event_date
    GROUP BY ob.order_id
)
SELECT
    ob.order_id,
    ob.user_id,
    ob.order_total,
    ob.currency,
    ob.payment_method,
    aa.account_age_days,
    ob.num_items,
    ob.num_distinct_products,
    h.previous_completed_purchases,
    h.previous_returns,
    h.previous_chargebacks,
    h.historical_avg_order_value,
    api.avg_days_between_purchases,
    h.days_since_last_purchase,
    h.user_return_rate,
    ua.user_session_count_7d,
    ua.user_page_views_7d,
    ua.user_cart_adds_7d,
    ua.user_cart_removals_7d,
    ua.user_searches_7d,
    opd.primary_device,
    opc.primary_category,
    ob.billing_street,
    ob.billing_city,
    ob.billing_state,
    ob.billing_zip_code,
    ob.billing_country,
    ob.shipping_street,
    ob.shipping_city,
    ob.shipping_state,
    ob.shipping_zip_code,
    ob.shipping_country
FROM order_base ob
LEFT JOIN account_age aa ON ob.user_id = aa.user_id
LEFT JOIN historical h ON ob.order_id = h.transaction_id
LEFT JOIN avg_purchase_interval api ON ob.user_id = api.user_id
LEFT JOIN user_activity_7d ua ON ob.user_id = ua.user_id AND ob.event_date = ua.event_date
LEFT JOIN order_primary_device opd ON ob.order_id = opd.order_id
LEFT JOIN order_primary_category opc ON ob.order_id = opc.order_id
""")

v_order_features.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}v_order_features")
# v_order_features.write.format("csv").option("header", "true").mode("overwrite").save(f"{gold_dir}v_order_features_csv")

print(f"v_order_features saved to: {gold_dir}v_order_features")
# print(f"v_order_features CSV saved to: {gold_dir}v_order_features_csv")
print(f"Total orders: {v_order_features.count()}")
print(f"Unique order_ids: {v_order_features.select('order_id').distinct().count()}")
display(v_order_features)
# display(spark.read.format("delta").load(f"{gold_dir}v_order_features").limit(5))